# Class 3: Adaptive Capacity

## Objective
**Score adaptive capacity as High (3), Medium (2), or Low (1)** based on when the building was constructed relative to flood regulation adoption dates.

We will assess how well buildings are prepared to withstand or adapt to flooding by examining their construction dates relative to important flood regulation milestones in Asheville/Buncombe County.

---

## What is Adaptive Capacity?

**Adaptive capacity** is a measure of how well an asset (in this case, a building) can withstand or adapt to a hazard (flooding). Think of it like:

- A person in good health is more *adaptive* to illness than someone with chronic disease
- A newer car with modern safety features is more *adaptive* to accidents than a 1970s car
- A building designed with flood-resistant features is more *adaptive* to flooding than an older building

### The Key Question for Flooding:
**When was this building built?** If it was built after flood regulations were implemented, the building was designed with those regulations in mind (elevated foundations, flood-resistant materials, etc.). If it was built before, it probably was not.

---

## Why Does "Year Built" Matter?

Building codes evolve over time as we learn more about hazards. Flood-resistant building practices were not widespread until the **National Flood Insurance Program (NFIP)** started requiring them in the 1970s-1980s.

### Key Dates for Asheville/Buncombe County:

| Period | Regulation Status | Adaptive Capacity |
|--------|------------------|-------------------|
| **Before 1980** | Built before NFIP participation — no flood-resistant design required | **LOW (1)** |
| **1981–2010** | Built after initial NFIP rules — BFE* of 1 foot above base flood elevation | **MEDIUM (2)** |
| **After 2010** | Built after stricter current rules — BFE of 2 feet above base flood elevation | **HIGH (3)** |

*BFE = Base Flood Elevation (the expected water level during a 100-year flood)

### Special Case: Not Exposed Parcels
If a parcel is **not exposed to flood risk** (from Class 1), it automatically gets **HIGH (3)** adaptive capacity because there is no flood risk to adapt to.

---

## Learning Goals for This Class

By the end of this class, you will be able to:

1. **Understand** why building age relates to flood resilience
2. **Calculate** adaptive capacity scores based on regulatory dates
3. **Handle** missing data responsibly in GIS analysis
4. **Visualize** adaptive capacity on a map
5. **Summarize** results with statistical tables

Let's get started!


## Step 0: Setup

In this section, we mount Google Drive and install all required libraries. This is standard for any Google Colab notebook that needs to access files from your Drive.


In [ ]:
# === ENVIRONMENT SETUP ===
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
    print("Google Drive connected!")
except Exception:
    BASE_DIR = './'
    print("Running locally.")

DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GeoPackage file paths for chained loading
INPUT_GPKG = os.path.join(DATA_DIR, 'class_2_impact.gpkg')  # Load from Class 2
OUTPUT_GPKG = os.path.join(DATA_DIR, 'class_3_adaptive_capacity.gpkg')  # Save to Class 3
CLASS0_GPKG = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')  # Fallback for base layers

print(f"  Base directory: {BASE_DIR}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")

In [ ]:
# Install libraries we need for GIS analysis
!pip install geopandas fiona shapely pyproj requests folium seaborn contextily --quiet

In [ ]:
# Import all the libraries we'll use
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium import plugins
import os
import warnings
warnings.filterwarnings('ignore')

# Set up for nice-looking plots
plt.style.use('seaborn-v0_8-darkgrid')

## Step 1: Load Data

We'll load the parcels layer from the GeoPackage. This layer should have:
- **exposure** field from Class 1 (1 = exposed to flood zone, 0 = not exposed)
- **potential_impact_a1** and **potential_impact_a2** fields from Class 2 (how severe the impact would be if flooded, scored separately per asset group)
- **structyear** field (year the building was constructed) from NC OneMap
- **geometry** (the parcel boundaries as polygons)

Let's load it and check its contents.

In [ ]:
# List all layers in the GeoPackage to see what we have
import fiona
layers = fiona.listlayers(INPUT_GPKG)
print(f"Layers in {os.path.basename(INPUT_GPKG)}:")
for layer in layers:
    print(f"  - {layer}")

In [ ]:
# Load the parcels layer from the INPUT GeoPackage (from Class 2)
try:
    parcels = gpd.read_file(INPUT_GPKG, layer='parcels')
    print(f"✓ Successfully loaded 'parcels' layer from Class 2")
    print(f"  Total parcels: {len(parcels):,}")
    print(f"  Columns: {list(parcels.columns)}")
except Exception as e:
    print(f"✗ Error loading parcels: {e}")
    print(f"Attempting fallback from CLASS0_GPKG: {CLASS0_GPKG}")
    try:
        parcels = gpd.read_file(CLASS0_GPKG, layer='parcels')
        print(f"✓ Successfully loaded from fallback")
    except Exception as e2:
        print(f"✗ Fallback also failed: {e2}")
        parcels = None

In [ ]:
# Check that the required fields exist
required_fields = ['exposure_a1', 'exposure_a2', 'potential_impact_a1', 'potential_impact_a2', 'sourcedate']
missing_fields = [f for f in required_fields if f not in parcels.columns]

if missing_fields:
    print(f"⚠ WARNING: Missing fields: {missing_fields}")
    print("Make sure you completed Classes 1 and 2 before this class!")
else:
    print(f"✓ All required fields present")

# Display the first few rows to understand the data structure
print("\nFirst 5 rows of data:")
print(parcels[['exposure_a1', 'exposure_a2', 'potential_impact_a1', 'potential_impact_a2', 'sourcedate']].head())

## Step 2: Explore the structyear Field

**What is sourcedatx?**

sourcedatx is a field from NC OneMap that we can use as a proxy for the year a building was constructed. It's based on property tax records and public data. Some parcels may have missing values (NULL) if the data was not recorded or the building is very new. and it should be best data to proxy the year a structure is built.

Let's explore this field to understand what we're working with:
- What's the range of years?
- Are there missing values?
- What does the distribution look like?


In [ ]:
# Convert saledatetx from string to datetime, then extract year
parcels['source_dt'] = pd.to_datetime(parcels['sourcedatx'], errors='coerce')
parcels['structyear'] = parcels['source_dt'].dt.year

# Get basic statistics about sale year
print("saledatetx Summary Statistics:")
print("=" * 50)
print(f"  Minimum year:     {parcels['structyear'].min():.0f}")
print(f"  Maximum year:     {parcels['structyear'].max():.0f}")
print(f"  Mean year:        {parcels['structyear'].mean():.1f}")
print(f"  Median year:      {parcels['structyear'].median():.0f}")
print()

# Count missing or zero values
null_count = parcels['structyear'].isna().sum()
zero_count = (parcels['structyear'] == 0).sum()
valid_count = len(parcels) - null_count - zero_count

print(f"  Valid years:      {valid_count:,} parcels ({100*valid_count/len(parcels):.1f}%)")
print(f"  Null/Missing:     {null_count:,} parcels ({100*null_count/len(parcels):.1f}%)")
print(f"  Zero values:      {zero_count:,} parcels ({100*zero_count/len(parcels):.1f}%)")
print(f"  Total parcels:    {len(parcels):,}")

In [ ]:
# Create a histogram showing the distribution of building years
fig, ax = plt.subplots(figsize=(14, 6))

# Filter out zero and null values for the histogram
valid_years = parcels[parcels['structyear'] > 0]['structyear'].dropna()

# Create the histogram
ax.hist(valid_years, bins=50, color='steelblue', edgecolor='black', alpha=0.7)

# Add vertical lines for the key regulation dates
ax.axvline(1980, color='red', linestyle='--', linewidth=2, label='1980: Start of NFIP Era')
ax.axvline(2010, color='orange', linestyle='--', linewidth=2, label='2010: Current Regulations')

# Labels and formatting
ax.set_xlabel('Year Built', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Buildings', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Building Construction Years\nAsheville/Buncombe County Parcels',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3)

# Format y-axis as thousands with commas
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

plt.tight_layout()
plt.show()

print(f"This histogram shows when buildings were built.")
print(f"The vertical lines mark the key flood regulation dates.")

### What Does the Histogram Tell Us?

Look at the histogram above:

1. **Before 1980 (red line):** How many buildings were built before flood regulations?
2. **1980-2010 (between red and orange lines):** How many were built under early regulations?
3. **After 2010 (orange line):** How many were built under current stricter rules?

Buildings built after each line should be progressively more flood-resistant. This is the basis for our adaptive capacity scoring!


## Step 3: Handle Missing Data

**The Problem:** Some parcels don't have a year built recorded in structyear. We can't calculate adaptive capacity without knowing when the building was constructed.

**Our Strategy:**
- For missing or zero structyear values, we'll assign **MEDIUM (2)** adaptive capacity
- This is a reasonable default: we assume a building of unknown age has moderate flood-resistant features
- We'll document this decision clearly so anyone using our results understands what we did

**Why Medium and not Low?**
- If we assumed Low, we'd be too pessimistic about buildings we know nothing about
- If we assumed High, we'd be too optimistic
- Medium reflects uncertainty: unknown buildings are a mixed bag

This is a common practice in risk assessment called "conservative uncertainty handling."


In [ ]:
# Check for missing structyear values before we calculate adaptive capacity
print("Missing structyear Assessment:")
print("=" * 50)

# Count the nulls and zeros
null_mask = parcels['structyear'].isna()
zero_mask = parcels['structyear'] == 0
missing_mask = null_mask | zero_mask

print(f"Parcels with NULL structyear:     {null_mask.sum():,}")
print(f"Parcels with structyear = 0:      {zero_mask.sum():,}")
print(f"Total missing/invalid:             {missing_mask.sum():,} ({100*missing_mask.sum()/len(parcels):.2f}%)")
print()
print("Decision: We will assign MEDIUM (2) adaptive capacity to all missing values.")
print("This reflects that buildings of unknown age are not automatically weak or strong.")

## Step 4: Calculate Adaptive Capacity (Separately for Each Asset Group)

We calculate adaptive capacity **separately** for Asset 1 and Asset 2 parcels.
Only parcels belonging to an asset group receive scores for that group.

| Condition | Adaptive Capacity Score |
|-----------|-------------------------|
| Parcel NOT exposed to flooding (exposure = 0) | **HIGH (3)** |
| Missing or zero structyear | **MEDIUM (2)** |
| Built before 1980 | **LOW (1)** |
| Built 1981-2010 | **MEDIUM (2)** |
| Built after 2010 | **HIGH (3)** |

This produces two columns: `adaptive_capacity_a1` and `adaptive_capacity_a2`.

In [ ]:
# Define the function to calculate adaptive capacity
def calculate_adaptive_capacity(row, exposure_col):
    """
    Calculate adaptive capacity score based on exposure and year built.

    Returns:
      3 = HIGH: Not exposed, or built after 2010
      2 = MEDIUM: Exposed + missing data, or built 1981-2010
      1 = LOW: Exposed + built before 1980
    """
    if row[exposure_col] == 0:
        return 3
    if pd.isna(row['structyear']) or row['structyear'] == 0:
        return 2
    year = row['structyear']
    if year < 1980:
        return 1  # LOW
    elif year <= 2010:
        return 2  # MEDIUM
    else:
        return 3  # HIGH

# Ensure asset flags exist
for col in ['is_asset_1', 'is_asset_2']:
    if col not in parcels.columns:
        print(f"WARNING: {col} not found — please run Class 1 first")

# Ensure exposure_a1/exposure_a2 exist
if 'exposure_a1' not in parcels.columns:
    parcels['exposure_a1'] = parcels['exposure'].where(parcels['is_asset_1'] == 1, 0).astype(int)
if 'exposure_a2' not in parcels.columns:
    parcels['exposure_a2'] = parcels['exposure'].where(parcels['is_asset_2'] == 1, 0).astype(int)

# Calculate adaptive capacity per asset group using its own exposure column
parcels['_ac_raw_a1'] = parcels.apply(lambda row: calculate_adaptive_capacity(row, 'exposure_a1'), axis=1)
parcels['_ac_raw_a2'] = parcels.apply(lambda row: calculate_adaptive_capacity(row, 'exposure_a2'), axis=1)

# Only parcels in the asset group get scored; others get 0
parcels['adaptive_capacity_a1'] = parcels['_ac_raw_a1'].where(parcels['is_asset_1'] == 1, 0).astype(int)
parcels['adaptive_capacity_a2'] = parcels['_ac_raw_a2'].where(parcels['is_asset_2'] == 1, 0).astype(int)
parcels.drop(columns=['_ac_raw_a1', '_ac_raw_a2'], inplace=True)

print("Adaptive capacity calculated separately for each asset group!")

ac_labels = {3: 'HIGH', 2: 'MEDIUM', 1: 'LOW', 0: 'Not scored'}
for col, label in [('adaptive_capacity_a1', 'Asset 1'), ('adaptive_capacity_a2', 'Asset 2')]:
    asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
    in_group = parcels[parcels[asset_flag] == 1]
    print(f"\n{label} ({len(in_group):,} parcels):")
    for score in [3, 2, 1]:
        count = (in_group[col] == score).sum()
        pct = 100 * count / len(in_group) if len(in_group) > 0 else 0
        print(f"  {ac_labels[score]:8} ({score}): {count:7,} parcels  {pct:5.1f}%")

In [ ]:
# Verify the calculation by looking at examples from each asset group
print("Sample calculations from Asset 1:")
print("=" * 80)
a1_sample = parcels[parcels['is_asset_1'] == 1][['exposure_a1', 'structyear', 'adaptive_capacity_a1']].head(5)
a1_sample.columns = ['Exposure', 'Year Built', 'AC Score']
print(a1_sample.to_string())

print("\nSample calculations from Asset 2:")
print("=" * 80)
a2_sample = parcels[parcels['is_asset_2'] == 1][['exposure_a2', 'structyear', 'adaptive_capacity_a2']].head(5)
a2_sample.columns = ['Exposure', 'Year Built', 'AC Score']
print(a2_sample.to_string())

## Step 5: Summary Statistics

Let's see what our adaptive capacity results look like. We'll calculate:
- How many parcels are at each adaptive capacity level?
- What's the total property value for each level?
- How does adaptive capacity relate to asset type?


In [ ]:
# Distribution of Adaptive Capacity per asset group
print("Distribution of Adaptive Capacity:")
print("=" * 60)

ac_labels = {3: 'HIGH', 2: 'MEDIUM', 1: 'LOW'}

for col, label in [('adaptive_capacity_a1', 'Asset 1'), ('adaptive_capacity_a2', 'Asset 2')]:
    asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
    in_group = parcels[parcels[asset_flag] == 1]
    print(f"\n{label} ({len(in_group):,} parcels):")
    for ac_level in [3, 2, 1]:
        count = (in_group[col] == ac_level).sum()
        pct = 100 * count / len(in_group) if len(in_group) > 0 else 0
        bar = '█' * int(pct / 2)
        print(f"  {ac_labels[ac_level]:8} ({ac_level}): {count:7,} parcels  {pct:5.1f}%  {bar}")

In [ ]:
# Parcel value by Adaptive Capacity per asset group
if 'parval' in parcels.columns:
    print("Total Parcel Value by Adaptive Capacity:")
    print("=" * 60)
    
    for col, label in [('adaptive_capacity_a1', 'Asset 1'), ('adaptive_capacity_a2', 'Asset 2')]:
        asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
        in_group = parcels[parcels[asset_flag] == 1]
        print(f"\n{label}:")
        for ac in [3, 2, 1]:
            subset = in_group[in_group[col] == ac]
            total = subset['parval'].sum()
            mean = subset['parval'].mean() if len(subset) > 0 else 0
            print(f"  AC={ac}: {len(subset):,} parcels, total=${total:,.0f}, avg=${mean:,.0f}")
else:
    print("Note: 'parval' field not found.")

In [ ]:
# Calculate structure_value as parval - landval (clamped to 0 minimum)
parcels['structure_value'] = (parcels['parval'] - parcels['landval']).clip(lower=0)

neg_count = ((parcels['parval'] - parcels['landval']) < 0).sum()
if neg_count > 0:
    print(f"Note: {neg_count:,} parcels had negative structure values (landval > parval), clamped to 0")

# Summarize structure value by adaptive capacity for each asset group
print("\nStructure Value by Adaptive Capacity (Asset 1 and Asset 2):")
print("=" * 60)

for col, label in [('adaptive_capacity_a1', 'Asset 1'), ('adaptive_capacity_a2', 'Asset 2')]:
    asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
    in_group = parcels[parcels[asset_flag] == 1]
    print(f"\n{label}:")
    for score in [3, 2, 1]:
        subset = in_group[in_group[col] == score]
        total_val = subset['structure_value'].sum()
        print(f"  AC={score}: {len(subset):,} parcels, structure value: ${total_val:,.0f}")

In [ ]:
# Cross-tabulation: Adaptive Capacity vs Exposure per asset group
print("Adaptive Capacity vs Exposure to Flooding:")
print("=" * 60)

for col, label, exp_col in [
    ('adaptive_capacity_a1', 'Asset 1', 'exposure_a1'),
    ('adaptive_capacity_a2', 'Asset 2', 'exposure_a2'),
]:
    asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
    in_group = parcels[parcels[asset_flag] == 1]
    print(f"\n{label}:")
    ct = pd.crosstab(in_group[col], in_group[exp_col], margins=True)
    ct.index = [f'AC={i}' if i != 'All' else 'Total' for i in ct.index]
    ct.columns = [f'Exp={c}' if c != 'All' else 'Total' for c in ct.columns]
    print(ct.to_string())

## Step 6: Map Adaptive Capacity

Now let's visualize adaptive capacity on a map. We'll use:
- **RED** for LOW (1) adaptive capacity — older buildings with limited flood resilience
- **YELLOW** for MEDIUM (2) adaptive capacity — moderate flood resilience
- **GREEN** for HIGH (3) adaptive capacity — newer, more resilient buildings

The map will help us see where vulnerable buildings are concentrated geographically.


In [ ]:
# Map of adaptive capacity per asset group
ac_colors = {0: '#555555', 1: '#FF5252', 2: '#FEE090', 3: '#1A9850'}
ac_labels = {0: 'Not Scored', 1: 'Low (1)', 2: 'Medium (2)', 3: 'High (3)'}

fig, axes = plt.subplots(1, 2, figsize=(20, 8), facecolor='#2b2b2b')

for ax, col, label in zip(axes, ['adaptive_capacity_a1', 'adaptive_capacity_a2'], ['Asset 1', 'Asset 2']):
    ax.set_facecolor('#2b2b2b')
    parcels.plot(ax=ax, color='#3a3a3a', edgecolor='#444444', linewidth=0.2, alpha=0.4)
    
    for score in [1, 2, 3]:
        subset = parcels[parcels[col] == score]
        if len(subset) > 0:
            subset.plot(ax=ax, color=ac_colors[score], edgecolor='black', linewidth=0.3,
                       alpha=0.8, label=f'{ac_labels[score]} (n={len(subset)})')
    
    ax.set_title(f'Adaptive Capacity: {label}', fontsize=14, fontweight='bold', color='white')
    ax.legend(loc='upper right', fontsize=9, facecolor='#3a3a3a', edgecolor='#555555', labelcolor='white')
    ax.set_axis_off()

plt.tight_layout()
plt.show()

## Export Adaptive Capacity Map to PNG

We'll create a high-resolution PNG map showing the adaptive capacity assessment with all layers properly styled and ordered.

In [ ]:
import contextily as ctx
from matplotlib.patches import Patch

parcels_wm = parcels.to_crs(epsg=3857)
ac_colors = {1: '#FF5252', 2: '#FEE090', 3: '#1A9850'}

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

for ax, col, label in zip(axes, ['adaptive_capacity_a1', 'adaptive_capacity_a2'], ['Asset 1', 'Asset 2']):
    parcels_wm.plot(ax=ax, facecolor='none', edgecolor='#888888', linewidth=0.3)
    for score in [1, 2, 3]:
        subset = parcels_wm[parcels_wm[col] == score]
        if len(subset) > 0:
            subset.plot(ax=ax, facecolor=ac_colors[score], edgecolor='none', alpha=0.85)
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom='auto')
    ax.set_axis_off()
    ax.set_title(f'Adaptive Capacity: {label}', fontsize=14, fontweight='bold', pad=10)

png_path = os.path.join(OUTPUT_DIR, 'adaptive_capacity_map.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"Map exported to: {png_path}")

## Step 7: Save Results

We save the updated parcels layer with separate adaptive capacity scores (`adaptive_capacity_a1`, `adaptive_capacity_a2`)
and the clamped `structure_value` field. Class 4 will use these to calculate vulnerability for each asset group.

In [ ]:
try:
    import fiona
    import sqlite3
    
    # Save parcels layer first (creates new GeoPackage)
    parcels.to_file(OUTPUT_GPKG, layer='parcels', driver='GPKG', mode='w')
    print(f"✓ Saved parcels layer with adaptive_capacity ({len(parcels)} features)")
    
    # Copy forward all other layers from the input GeoPackage
    if os.path.exists(INPUT_GPKG):
        input_layers = fiona.listlayers(INPUT_GPKG)
        for layer_name in input_layers:
            if layer_name == 'parcels':
                continue  # Already saved updated version
            try:
                layer_data = gpd.read_file(INPUT_GPKG, layer=layer_name)
                layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                print(f"✓ Copied layer: {layer_name} ({len(layer_data)} features)")
            except Exception as e:
                print(f"  Note: Could not copy layer '{layer_name}' as spatial: {e}")
        
        # Also copy any non-spatial tables (like 'summary') via sqlite3
        try:
            conn_in = sqlite3.connect(INPUT_GPKG)
            conn_out = sqlite3.connect(OUTPUT_GPKG)
            cursor = conn_in.cursor()
            # Get all tables that aren't in fiona's layer list and aren't system tables
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
            all_tables = [row[0] for row in cursor.fetchall()]
            system_tables = ['gpkg_contents', 'gpkg_geometry_columns', 'gpkg_spatial_ref_sys',
                            'gpkg_ogr_contents', 'gpkg_tile_matrix', 'gpkg_tile_matrix_set',
                            'sqlite_sequence', 'gpkg_extensions', 'gpkg_metadata',
                            'gpkg_metadata_reference']
            for table in all_tables:
                if table in system_tables or table in input_layers or table.startswith('rtree_') or table.startswith('trigger_'):
                    continue
                try:
                    df = pd.read_sql(f'SELECT * FROM "{table}"', conn_in)
                    if len(df) > 0:
                        df.to_sql(table, conn_out, if_exists='replace', index=False)
                        print(f"✓ Copied non-spatial table: {table} ({len(df)} rows)")
                except Exception:
                    pass
            conn_in.close()
            conn_out.close()
        except Exception:
            pass
    
    # Also ensure base layers from Class 0 are included (flood_zones, buildings, study_area)
    # These may not be in INPUT_GPKG if earlier classes didn't carry them forward
    if os.path.exists(CLASS0_GPKG):
        try:
            import fiona as _fiona
            # Get layers already written to output
            output_layers = _fiona.listlayers(OUTPUT_GPKG)
            # Get layers available in Class 0
            class0_layers = _fiona.listlayers(CLASS0_GPKG)
            # Copy any missing layers
            for layer_name in class0_layers:
                if layer_name not in output_layers:
                    try:
                        layer_data = gpd.read_file(CLASS0_GPKG, layer=layer_name)
                        layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                        print(f"✓ Added base layer from Class 0: {layer_name} ({len(layer_data)} features)")
                    except Exception as e:
                        print(f"  Note: Could not copy base layer '{layer_name}': {e}")
        except Exception:
            pass
    
    print(f"\n✓ All data saved to: {OUTPUT_GPKG}")
except Exception as e:
    print(f"✗ Error saving to GeoPackage: {e}")
    print("\nAlternative: Save to a new GeoPackage file")
    alt_path = os.path.join(DATA_DIR, 'class_3_adaptive_capacity_backup.gpkg')
    parcels.to_file(alt_path, layer='parcels', driver='GPKG')
    print(f"✓ Saved to: {alt_path}")

In [ ]:
# Verify the save by reading it back
try:
    verify = gpd.read_file(OUTPUT_GPKG, layer='parcels')
    for col in ['adaptive_capacity_a1', 'adaptive_capacity_a2']:
        if col in verify.columns:
            print(f"✓ '{col}' found in saved file")
            print(f"  Value counts: {verify[col].value_counts().sort_index(ascending=False).to_dict()}")
        else:
            print(f"⚠ '{col}' NOT found in saved file")
except Exception as e:
    print(f"Error verifying save: {e}")

## Step 8: Interpretation & Next Steps

### What We've Accomplished

We've scored **adaptive capacity** — a measure of how well buildings can withstand flooding — based on:

1. **Building construction date** relative to flood regulation dates
2. **Exposure status** from Class 1
3. **Data quality** (handling missing structyear values responsibly)

### Key Takeaways

- **Older buildings (pre-1980)** in flood zones have LOW adaptive capacity. They were built before anyone thought about flood-resistant design.
- **Medium-age buildings (1981-2010)** have MEDIUM adaptive capacity. They follow basic flood regulations.
- **Newer buildings (post-2010)** have HIGH adaptive capacity. They meet current stricter standards.
- **Not-exposed buildings** always have HIGH adaptive capacity — even if old — because they're not at risk.

### Next Steps

In **Class 4: Risk Score**, we'll combine:
- **Hazard** (flood probability from Class 1)
- **Exposure** (buildings in flood zones)
- **Vulnerability** (potential impact from Class 2)
- **Adaptive Capacity** (resilience we calculated here)

...to create a final **RISK SCORE** that tells us which properties are most at risk.

### Questions to Discuss

1. Why do you think adaptive capacity based on building age makes sense?
2. Are there any other factors that might affect a building's ability to withstand flooding (besides age)?
3. Why did we assign MEDIUM capacity to buildings with missing construction dates?

---


## Appendix A: QGIS Equivalent Expression

If you wanted to calculate adaptive capacity in **QGIS** instead of Python, you could use the **Field Calculator** with this expression:

```qgis
CASE
  WHEN "exposure" = 0 THEN 3
  WHEN "structyear" IS NULL OR "structyear" = 0 THEN 2
  WHEN "structyear" < 1980 THEN 1
  WHEN "structyear" >= 1981 AND "structyear" <= 2010 THEN 2
  WHEN "structyear" > 2010 THEN 3
  ELSE 2
END
```

### Steps in QGIS:

1. Load the parcels layer
2. Open **Layer → Fields → New Field** (or double-click to edit existing layer)
3. Create field named **adaptive_capacity** (type: Integer)
4. Right-click the field → **Edit Field**
5. Switch to **Field Calculator** mode
6. Paste the expression above
7. Click **OK** to apply to all rows

The logic is identical to our Python function!


> **Note:** You will need to repeat these steps for each asset group. The field names above use the Asset 1 suffix (`_a1`). For Asset 2, replace `_a1` with `_a2` in all field references.

## Appendix B: ArcGIS Pro Equivalent

If you wanted to calculate adaptive capacity in **ArcGIS Pro**, you could use **Calculate Field** with a Python expression.

### ArcGIS Pro Field Calculator Expression:

```python
def calc_adaptive_capacity(exposure, structyear):
    """Calculate adaptive capacity score"""
    if exposure == 0:
        return 3  # Not exposed
    if structyear is None or structyear == 0:
        return 2  # Missing data
    if structyear < 1980:
        return 1  # Low - pre-1980
    elif structyear <= 2010:
        return 2  # Medium - 1981-2010
    else:
        return 3  # High - post-2010

calc_adaptive_capacity(!exposure!, !structyear!)
```

### Steps in ArcGIS Pro:

1. Load the parcels feature class
2. Right-click the **adaptive_capacity** field → **Calculate**
3. Set **Parser** to **Python 3**
4. Define the function above in the expression area
5. Call it with: `calc_adaptive_capacity(!exposure!, !structyear!)`
6. Click **OK** to apply

This approach is best practice in ArcGIS because it's readable and reusable!


> **Note:** You will need to repeat these steps for each asset group. The field names above use the Asset 1 suffix (`_a1`). For Asset 2, replace `_a1` with `_a2` in all field references.

## Conclusion

You've successfully scored adaptive capacity **separately for Asset 1 and Asset 2**.

**You now understand:**
- Why building age matters for flood resilience
- How to handle missing data responsibly
- How to calculate scores based on regulatory thresholds
- How Asset 1 and Asset 2 are scored independently

**Columns added:** `adaptive_capacity_a1`, `adaptive_capacity_a2`, `structure_value` (clamped to 0 minimum)

Next up: **Class 4 (Vulnerability)** — combining potential impact and adaptive capacity into a vulnerability score, separately for each asset group.